# 03 — Multimodal Fusion (Image + Text)
Loads trained ResNet50 backbone, pairs images with word-pool text descriptions,
then trains and compares two fusion strategies:
- **Splice Fusion** — concatenate image & text features → MLP
- **Weighted Fusion** — weighted sum of image & text features → MLP

> **Prerequisite:** Run `02_image_models.ipynb` first so ResNet50 checkpoint exists.

## 1. Mount Drive and load dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/intel_data'):
    print('Copying dataset from Drive...')
    os.system('cp -r /content/drive/MyDrive/ContentRecognition/image_dataset /content/intel_data')
    print('Done!')

## 2. Imports

In [ ]:
import os, random, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import DistilBertTokenizer, DistilBertModel
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 3. Config

In [ ]:
CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE  = 32
EPOCHS      = 10
BASE_DIR    = '/content/drive/MyDrive/ContentRecognition'
CKPT_DIR    = f'{BASE_DIR}/checkpoints/image'
RESULTS_DIR = f'{BASE_DIR}/results/image'

# Path to the ResNet50 checkpoint trained in 02_image_models.ipynb
RESNET_CKPT = f'{CKPT_DIR}/ResNet50.pth'

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(RESNET_CKPT), (
    f'ResNet50 checkpoint not found at {RESNET_CKPT}. '
    'Run 02_image_models.ipynb first!'
)
print('Config OK. ResNet50 checkpoint found.')

## 4. Word pools
Each class has domain-specific words. During training a description is built from
3 correct-class words + 2 words from another class + 1 shared ambiguous word.
This simulates real-world noisy captions and stops the text alone from being
a perfect signal — forcing the model to actually fuse both modalities.

In [ ]:
CLASS_WORD_POOLS = {
    'buildings': ['tall', 'urban', 'building', 'architecture', 'city',
                  'construction', 'structure', 'skyscraper', 'wall', 'concrete'],
    'forest'   : ['dense', 'green', 'trees', 'forest', 'nature',
                  'woodland', 'leaves', 'jungle', 'wild', 'canopy'],
    'glacier'  : ['ice', 'cold', 'frozen', 'glacier', 'snow',
                  'arctic', 'frost', 'icy', 'crevasse', 'blizzard'],
    'mountain' : ['rocky', 'mountain', 'peak', 'summit', 'cliff',
                  'highland', 'steep', 'altitude', 'ridge', 'valley'],
    'sea'      : ['ocean', 'water', 'waves', 'beach', 'coastal',
                  'shore', 'marine', 'horizon', 'sea', 'tide'],
    'street'   : ['road', 'street', 'traffic', 'pavement', 'sidewalk',
                  'lane', 'intersection', 'pedestrian', 'crosswalk', 'signal'],
}

SHARED_WORDS = ['outdoor', 'light', 'dark', 'wide', 'large',
                'natural', 'open', 'landscape', 'scene', 'environment']


def generate_description(cls_name):
    """3 class words + 2 from another class + 1 shared ambiguous word."""
    own_words   = random.sample(CLASS_WORD_POOLS[cls_name], 3)
    other_cls   = random.choice([c for c in CLASS_NAMES if c != cls_name])
    other_words = random.sample(CLASS_WORD_POOLS[other_cls], 2)
    shared      = random.sample(SHARED_WORDS, 1)
    words       = own_words + other_words + shared
    random.shuffle(words)
    return ' '.join(words)


# Quick sanity check
print('Sample descriptions:')
for c in CLASS_NAMES:
    print(f'  {c:12} → "{generate_description(c)}"')

## 5. Load DistilBERT (frozen text encoder)

In [ ]:
tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)

# Freeze — BERT is only a feature extractor, not fine-tuned
for p in bert_model.parameters():
    p.requires_grad = False
bert_model.eval()


@torch.no_grad()
def get_text_features(text):
    """Return [CLS] token embedding  →  shape (batch, 768)."""
    tokens = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=32
    ).to(device)
    out = bert_model(**tokens)
    return out.last_hidden_state[:, 0, :]   # (batch, 768)


print('DistilBERT loaded and frozen!')

## 6. Load ResNet50 as frozen image feature extractor

In [ ]:
# Rebuild the same architecture used during training
resnet_full = models.resnet50(weights=None)
resnet_full.fc = nn.Sequential(
    nn.Linear(resnet_full.fc.in_features, 256),
    nn.ReLU(), nn.Dropout(0.4),
    nn.Linear(256, NUM_CLASSES)
)
resnet_full.load_state_dict(torch.load(RESNET_CKPT, map_location=device))

# Strip the classification head — keep only the backbone (output: B×2048×1×1)
image_encoder = nn.Sequential(*list(resnet_full.children())[:-1]).to(device)
for p in image_encoder.parameters():
    p.requires_grad = False
image_encoder.eval()

print('ResNet50 backbone loaded and frozen!')
print('Image feature dim: 2048')

## 7. Dataset — image + generated text description

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


class ImageTextDataset(Dataset):
    """
    Returns (image_tensor, text_feature_768, label).
    Text is generated on-the-fly from word pools.
    During training: random description every call.
    During validation: fixed seed per index for reproducibility.
    """
    def __init__(self, root, transform, training=False):
        from torchvision.datasets import ImageFolder
        base = ImageFolder(root, transform=None)
        self.samples   = base.samples   # list of (path, label_idx)
        self.transform = transform
        self.training  = training

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        cls_name        = CLASS_NAMES[label]

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        if self.training:
            desc = generate_description(cls_name)
        else:
            # Fixed seed per sample → reproducible val descriptions
            rng_state = random.getstate()
            random.seed(idx)
            desc = generate_description(cls_name)
            random.setstate(rng_state)

        # Pre-compute text features here (inside Dataset)
        # squeeze(0): remove batch dim added by tokenizer
        text_feat = get_text_features(desc).squeeze(0).cpu()

        return image, text_feat, label


train_ds = ImageTextDataset(
    '/content/intel_data/seg_train/seg_train', train_transforms, training=True
)
val_ds   = ImageTextDataset(
    '/content/intel_data/seg_test/seg_test',   val_transforms,   training=False
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} samples  |  Val: {len(val_ds):,} samples')

## 8. Fusion model definitions

Both fusion strategies share the same projectors (image 2048→512, text 768→512).
They differ only in *how* the two 512-d vectors are combined:

| Strategy | Formula |
|---|---|
| **Splice** | `concat([v, t])` → 1024-d → MLP |
| **Weighted** | `δ·v + (1-δ)·t` → 512-d → MLP |

In [ ]:
class SpliceFusionModel(nn.Module):
    """
    Concatenation (splice) fusion.
    image_feat (B,2048) + text_feat (B,768) → concat (B,1024) → classifier.
    """
    def __init__(self, num_classes=6):
        super().__init__()
        self.visual_proj = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(), nn.Dropout(0.3)
        )
        self.text_proj = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.3)
        )
        # 512 + 512 = 1024
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, img_feat, txt_feat):
        v     = self.visual_proj(img_feat)         # (B, 512)
        t     = self.text_proj(txt_feat)           # (B, 512)
        fused = torch.cat([v, t], dim=1)           # (B, 1024)
        return self.classifier(fused)


class WeightedFusionModel(nn.Module):
    """
    Weighted sum fusion.
    fused = delta * visual + (1-delta) * text  →  (B, 512)  →  classifier.
    delta=0.6 means we trust the image slightly more than the text.
    """
    def __init__(self, num_classes=6, delta=0.6):
        super().__init__()
        self.delta = delta
        self.visual_proj = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(), nn.Dropout(0.3)
        )
        self.text_proj = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.3)
        )
        # 512 only (no concat)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, img_feat, txt_feat):
        v     = self.visual_proj(img_feat)                      # (B, 512)
        t     = self.text_proj(txt_feat)                        # (B, 512)
        fused = self.delta * v + (1 - self.delta) * t          # (B, 512)
        return self.classifier(fused)


print('Both fusion models defined.')

In [ ]:
import math

class AttentionFusionModel(nn.Module):
    """
    Cross-Attention Fusion.

    Image features act as the QUERY  (what we want to know more about).
    Text  features act as KEY+VALUE  (the context that answers the query).

    The attention mechanism computes a dynamic weight that says:
    'given this image, which parts of the text description matter most?'
    The attended text vector is then added back to the image vector
    (residual connection) before classification.

    Architecture:
        image (2048) → visual_proj → (512)  = Q
        text  (768)  → text_proj   → (512)  = K, V

        attn_score = softmax( Q·Kᵀ / √512 )         scalar per sample
        attended   = attn_score * V                  (512,)
        fused      = Q + attended                    residual add (512,)
        out        = LayerNorm(fused) → MLP → 6
    """
    def __init__(self, num_classes=6, proj_dim=512):
        super().__init__()
        self.proj_dim = proj_dim
        self.scale    = math.sqrt(proj_dim)

        # Project image features → query space
        self.visual_proj = nn.Sequential(
            nn.Linear(2048, proj_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Project text features → key/value space
        self.text_proj = nn.Sequential(
            nn.Linear(768, proj_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Layer norm after residual add — stabilises training
        self.layer_norm = nn.LayerNorm(proj_dim)

        # Final classifier on the attended fused vector
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, img_feat, txt_feat):
        # ── Project both modalities to same space ─────────────────
        Q = self.visual_proj(img_feat)   # (B, 512)  — Query
        K = self.text_proj(txt_feat)     # (B, 512)  — Key
        V = K                            # (B, 512)  — Value = Key (self-contained text)

        # ── Scaled dot-product attention ─────────────────────────
        # Q·Kᵀ: for each sample, dot product of its image and text vectors
        # Shape: (B, 1) — one attention score per sample
        attn_score = torch.sum(Q * K, dim=1, keepdim=True) / self.scale  # (B, 1)
        attn_weight = torch.sigmoid(attn_score)   # bound to (0,1)
        # Note: sigmoid (not softmax) because we have only 1 "head" here.
        # attn_weight ≈ 1  →  text is very relevant to this image
        # attn_weight ≈ 0  →  text adds little, rely more on image alone

        # ── Attend: scale the value vector by attention weight ────
        attended = attn_weight * V       # (B, 512)

        # ── Residual connection + LayerNorm ───────────────────────
        # Adding attended text BACK to the image query (residual):
        # fused = image features + how much text contributes
        fused = self.layer_norm(Q + attended)  # (B, 512)

        return self.classifier(fused)    # (B, num_classes)


print('AttentionFusionModel defined.')
print('Params:', sum(p.numel() for p in AttentionFusionModel().parameters()), '(untrained)')

## 9. Shared fusion training function

In [ ]:
def train_fusion(model, model_name, epochs=EPOCHS):
    """Train a fusion model. Returns (model, history, best_acc, save_path)."""
    model      = model.to(device)
    save_path  = f'{CKPT_DIR}/{model_name}.pth'
    optimizer  = optim.Adam(model.parameters(), lr=1e-3)
    scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    criterion  = nn.CrossEntropyLoss()
    history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc   = 0.0

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*54}')
    print(f'  Training : {model_name}  |  Params: {trainable:,}')
    print(f'{"="*54}')

    for epoch in range(epochs):
        t0 = time.time()
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            loader = train_loader if phase == 'train' else val_loader

            running_loss, running_correct = 0.0, 0

            for images, text_feats, labels in loader:
                images, labels = images.to(device), labels.to(device)
                text_feats     = text_feats.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    # Extract image features via frozen ResNet50
                    img_feat = image_encoder(images)          # (B, 2048, 1, 1)
                    img_feat = img_feat.view(img_feat.size(0), -1)  # (B, 2048)

                    outputs = model(img_feat, text_feats)
                    loss    = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss    += loss.item() * images.size(0)
                running_correct += torch.sum(preds == labels)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc  = running_correct.double() / len(loader.dataset)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), save_path)

        print(
            f'  Epoch {epoch+1:02d}/{epochs} | '
            f'Train {history["train_acc"][-1]:.4f} | '
            f'Val {history["val_acc"][-1]:.4f} | '
            f'{time.time()-t0:.1f}s'
        )

    print(f'  ✅ Best Val Acc: {best_acc:.4f}')
    return model, history, best_acc.item(), save_path


print('train_fusion() defined.')

## 10. Train Splice Fusion

In [ ]:
splice_model = SpliceFusionModel(num_classes=NUM_CLASSES)
splice_model, hist_splice, acc_splice, path_splice = train_fusion(
    splice_model, 'SpliceFusion'
)

In [ ]:
attention_model = AttentionFusionModel(num_classes=NUM_CLASSES)
attention_model, hist_attention, acc_attention, path_attention = train_fusion(
    attention_model, 'AttentionFusion'
)

## 11. Train Weighted Fusion (δ = 0.6)

In [ ]:
weighted_model = WeightedFusionModel(num_classes=NUM_CLASSES, delta=0.6)
weighted_model, hist_weighted, acc_weighted, path_weighted = train_fusion(
    weighted_model, 'WeightedFusion'
)

## 12. Evaluate both models

In [ ]:
def evaluate_fusion(model, save_path, model_name):
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval().to(device)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, text_feats, labels in val_loader:
            images     = images.to(device)
            text_feats = text_feats.to(device)

            img_feat = image_encoder(images)
            img_feat = img_feat.view(img_feat.size(0), -1)

            outputs  = model(img_feat, text_feats)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(f'\n{"="*54}')
    print(f'  {model_name} — Classification Report')
    print(f'{"="*54}')
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
    return all_preds, all_labels


preds_splice,   labels_ = evaluate_fusion(splice_model,   path_splice,   'Splice Fusion')
preds_weighted, _       = evaluate_fusion(weighted_model, path_weighted, 'Weighted Fusion')
preds_attention, labels_attention = evaluate_fusion(
    attention_model, path_attention, 'Attention Fusion'
)

## 13. Plots — compare both fusion strategies

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

COLORS = {
    'Splice Fusion'   : '#8E44AD',
    'Weighted Fusion' : '#E67E22',
    'Attention Fusion': '#1ABC9C',
}

all_fusion_results = {
    'Splice Fusion'   : acc_splice,
    'Weighted Fusion' : acc_weighted,
    'Attention Fusion': acc_attention,
}

# ── Plot A: Accuracy bar comparison ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(all_fusion_results.keys())
accs  = [v * 100 for v in all_fusion_results.values()]
bars  = axes[0].bar(
    names, accs,
    color=[COLORS[n] for n in names],
    width=0.45
)
for bar, acc in zip(bars, accs):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        f'{acc:.2f}%',
        ha='center', va='bottom', fontweight='bold'
    )
axes[0].set_title('Fusion Strategy Comparison', fontweight='bold')
axes[0].set_ylabel('Validation Accuracy (%)')
axes[0].set_ylim([min(accs) - 3, 102])
axes[0].grid(axis='y', alpha=0.3)

# ── Plot B: Val accuracy curves all 3 ────────────────────────────
for hist, name in zip(
    [hist_splice, hist_weighted, hist_attention], names
):
    axes[1].plot(
        hist['val_acc'], label=name,
        color=COLORS[name], marker='o'
    )
axes[1].set_title('Validation Accuracy per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/all_fusion_comparison.png', dpi=150)
plt.show()

# ── Plot C: Train vs Val curves for attention model ───────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

axes2[0].plot(hist_attention['train_acc'], label='Train',
              color='#1ABC9C', marker='o')
axes2[0].plot(hist_attention['val_acc'],   label='Val',
              color='#1ABC9C', linestyle='--', marker='s')
axes2[0].set_title('Attention Fusion — Accuracy', fontweight='bold')
axes2[0].legend(); axes2[0].grid(True, alpha=0.3)
axes2[0].set_xlabel('Epoch'); axes2[0].set_ylabel('Accuracy')

axes2[1].plot(hist_attention['train_loss'], label='Train',
              color='#1ABC9C', marker='o')
axes2[1].plot(hist_attention['val_loss'],   label='Val',
              color='#1ABC9C', linestyle='--', marker='s')
axes2[1].set_title('Attention Fusion — Loss', fontweight='bold')
axes2[1].legend(); axes2[1].grid(True, alpha=0.3)
axes2[1].set_xlabel('Epoch'); axes2[1].set_ylabel('Loss')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/attention_fusion_curves.png', dpi=150)
plt.show()

# ── Plot D: Confusion matrices all 3 ─────────────────────────────
fig3, axes3 = plt.subplots(1, 3, figsize=(21, 6))
for ax, preds, name in zip(
    axes3,
    [preds_splice, preds_weighted, preds_attention],
    names
):
    cm = confusion_matrix(labels_, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Purples',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax
    )
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.suptitle('Fusion Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/all_fusion_confusion_matrices.png', dpi=150)
plt.show()

print('All plots saved!')

## 14. Final comparison — image-only vs fusion

In [ ]:
# Load image-only ResNet50 result to compare
# (These will be accurate if you ran 02_image_models.ipynb in the same session;
#  otherwise enter the printed best-val-acc values manually.)
# resnet_image_only_acc = 0.93  # example — update with your result

print(f'\n{"="*55}')
print(f'{"FUSION STRATEGY":<22} {"VAL ACCURACY":>15} {"WINNER":>12}')
print(f'{"="*55}')

best = max(all_fusion_results, key=all_fusion_results.get)
for name, acc in all_fusion_results.items():
    winner = '🏆' if name == best else ''
    print(f'{name:<22} {acc*100:>14.2f}%  {winner:>10}')

print(f'{"="*55}')
print(f'\nBest fusion strategy: {best}  ({all_fusion_results[best]*100:.2f}%)')

# Improvement over image-only ResNet50 baseline
resnet_baseline = 0.924   # update with your actual ResNet50 val acc from notebook 02
print(f'\nImprovement over ResNet50 (image-only) baseline:')
for name, acc in all_fusion_results.items():
    diff = (acc - resnet_baseline) * 100
    sign = '+' if diff >= 0 else ''
    print(f'  {name:<22} {sign}{diff:.2f}%')